# E3.8 · Resilience over perfection

**Function E — AI Governance for Agentic Systems → Running the Programme — the CISO Office**  ·  *Security of AI*

Builds on **[E3.7 · Building the capability](https://spbreed.github.io/cyber-commons/lessons/E3.7.html)**.

| | |
|---|---|
| Open-source tooling | — |
| Open-weight models | — |
| Frontier models | — |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off — and where a lesson involves a model, the same code calls an open-weight endpoint or a frontier API when you configure one.

## 1 · The hook

You will not prevent every failure of a probabilistic system, and a programme that promises to will be judged on that promise. Designing for recovery is both more honest and more defensible than designing for perfection.

> **At CyberTravels.** TripBot will fail sometimes, because it is probabilistic. A programme that promised otherwise will be judged on that promise; one designed to detect fast, contain small and recover cheaply will not.

## 2 · The framework

```
   perfection                    resilience
   +------------------+          +-------------------------+
   | prevent every    |          | detect fast             |
   | failure          |    vs    | contain small           |
   |                  |          | recover cheaply         |
   +------------------+          | accept a named loss     |
   judged on the promise         +-------------------------+

   for a probabilistic system only one of these is a promise you can keep
```

The last lesson, and the one that reframes everything before it.

You will not prevent every agentic failure. The systems are non-deterministic,
the attack surface is novel, and the change surface bypasses your change process
(D1.7). A programme judged on prevention is judged on something it cannot
deliver, and it will report success right up until the first real incident.

Judge it on three capabilities instead, each independently testable, none of
them prevention:

- **Notice** — drift and detections fire when behaviour changes (D1.4, D1.7).
- **Stop** — a tested mechanism halts it, measured in seconds (D2.7).
- **Recover** — the run is replayable and the scope is knowable (D2.3, D2.5).

A programme with all three survives a failure it did not predict, which is the
only kind that actually happens. Perfection would mean containment never fails.
Resilience means the other five steps work when it does.

## 3 · Demo — test the three capabilities, not the prevention

In [ ]:
import time, statistics
now = time.time()

# --- NOTICE ------------------------------------------------------------
BASELINE = {"read_file": 0.85, "search": 0.15}
TODAY    = {"read_file": 300, "search": 100, "run_shell": 400}
total = sum(TODAY.values())
mix = {k: v/total for k, v in TODAY.items()}
keys = set(mix) | set(BASELINE)
drift = sum(abs(mix.get(k,0) - BASELINE.get(k,0)) for k in keys)/2
new_tools = sorted(set(mix) - set(BASELINE))
notice = drift > 0.25 or bool(new_tools)
print(f"NOTICE   drift {drift:.3f}  new tools {new_tools}  → "
      f"{'detected' if notice else 'MISSED'}")

# --- STOP --------------------------------------------------------------
STOP = {"mechanism": "revoke the SPIFFE identity at the gateway",
        "measured_seconds": 12, "tested_days_ago": 41, "survives_restart": True}
stop_ok = (STOP["measured_seconds"] is not None and STOP["tested_days_ago"] <= 180
           and STOP["survives_restart"])
print(f"STOP     {STOP['measured_seconds']}s, tested {STOP['tested_days_ago']}d ago, "
      f"survives restart {STOP['survives_restart']}  → {'ready' if stop_ok else 'NOT READY'}")

# --- RECOVER -----------------------------------------------------------
RUN = {"prompts": ["fix SEC-4471"], "tool_results": ["contents…"],
       "model_version": "glm-4.6@2026-07-14", "seed": 42}
missing = [k for k, v in RUN.items() if not v and v != 0]
CHAIN = ["dana@corp", "orchestrator", "patch-agent"]
REACHED = {"dana@corp": ["repo-core"], "orchestrator": ["queue"],
           "patch-agent": ["repo-core","repo-payments"]}
scope = sorted({r for a in CHAIN for r in REACHED.get(a, [])})
recover = not missing and bool(scope)
print(f"RECOVER  replayable {not missing}, scope from the chain {scope}  → "
      f"{'ready' if recover else 'NOT READY'}")

## 4 · Where it breaks — the prevention-only programme

In [ ]:
PROGRAMMES = {
 "prevention only": {"notice": False, "stop": False, "recover": False,
                     "containment_asr": 0.0},
 "prevention + notice": {"notice": True, "stop": False, "recover": False,
                         "containment_asr": 0.0},
 "resilient": {"notice": True, "stop": True, "recover": True,
               "containment_asr": 0.0},
}
def incident_outcome(p, containment_failed=True):
    if not containment_failed:
        return "no incident", 0
    if not p["notice"]:
        return "undetected — found by a third party, weeks later", 720
    if not p["stop"]:
        return "detected, cannot halt it — damage continues while you improvise", 96
    if not p["recover"]:
        return "detected and halted, cannot say what was touched or why", 48
    return "detected, halted in seconds, scope known, run replayable", 6

print(f"{'programme':24s}{'containment ASR':>17}  outcome when containment fails")
print("-" * 96)
for name, p in PROGRAMMES.items():
    outcome, hours = incident_outcome(p)
    print(f"{name:24s}{p['containment_asr']:>17.0%}  {outcome}")
    print(f"{'':41s}elapsed to resolution: {hours}h")
print("\nAll three have a 0% attack success rate. On a prevention-only")
print("scorecard they are identical. They are not remotely identical.")

## 5 · The control — the game day that assumes containment failed

In [ ]:
def game_day(programme):
    """Assume the prevention worked until it didn't. Measure the other three."""
    results = {}
    results["notice"]  = (0.2, "drift alert fired") if programme["notice"] \
                         else (None, "no signal — nothing fired")
    results["stop"]    = (12, "identity revoked, survives restart") if programme["stop"] \
                         else (None, "no tested mechanism")
    results["recover"] = (6, "replayed the run, scope from the act chain") \
                         if programme["recover"] else (None, "cannot reconstruct")
    weakest = next((k for k, (v, _) in results.items() if v is None), None)
    return results, weakest

for name, p in PROGRAMMES.items():
    res, weakest = game_day(p)
    print(f"=== {name} ===")
    for cap, (val, note) in res.items():
        print(f"   {cap:9s}{(str(val) + 'h') if val is not None else 'FAIL':>7}  {note}")
    print(f"   weakest capability: {weakest or 'none — all three hold'}\n")

_, weakest = game_day(PROGRAMMES["resilient"])
assert weakest is None
print("The weakest capability is next quarter's plan. That is the whole")
print("programme-management loop, and it does not require predicting the attack.")

In [ ]:
# Close the curriculum: what you built, and what it is for.
BUILT = [
 ("A1-A3", "a control plane: planes, identity, containment"),
 ("B1",    "a 15-stage AppSec pipeline, ending in confirmed-by-exploitation severity"),
 ("B2",    "a harness whose verifier does not lie"),
 ("C1-C2", "the ability to attack it and to research it repeatably"),
 ("D1-D2", "the ability to notice, stop and recover"),
 ("E1-E3", "the ability to evidence all of it, and to decide"),
]
for track, what in BUILT:
    print(f"   {track:8s}{what}")
print("\nNone of it assumes a frontier-lab account, a vendor platform, or a")
print("budget. That was the point: shared defense is stronger defense, and a")
print("commons only works if everyone can actually run it.")

## What you just proved

Drift is detected with `run_shell` as a new tool, the stop mechanism is ready at 12 seconds tested 41 days ago, and the run is replayable with a four-resource scope. All three programmes show a 0% containment ASR yet resolve a failure in 720, 96 and 6 hours respectively. The game day identifies the weakest capability for each, and the resilient programme has none.

## Your turn

Run a game day that assumes containment failed. Measure notice, stop and recover as three separate numbers. The weakest one is next quarter's plan — and unlike a prevention target, you can actually reach it.

## Where this leaves you

**What you can do now.** A programme you can sequence, staff and defend: autonomy governed by level rather than by product list, one owner per thing, metrics that show control rather than activity, conditional approvals that are actually tracked, and a design that assumes failure and recovers.

**What you still cannot do.** Nothing here is finished, because none of it holds still. The models change, the patterns change, and the risks in A1 will not be the last fifteen. What you have is a method for the next set, not a solution to this one.

**Go back to A1.1 and draw your own system again. It will be a different picture from the one you drew before Function B, and the components you left off the first time are the ones worth your next quarter.**

---

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/E3.8.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/E3.8.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*